# 🎞️ ANIMA on Diffucore UI — Google Colab (free tier)

Run **ANIMA** (CircleStone Labs' Cosmos-Predict2 2B text-to-image DiT) through
**[Diffucore UI](https://github.com/nawka12/diffucore-ui)** — a clean, linear
workspace (no ComfyUI nodes), Apache-2.0, powered by the
**[Diffucore](https://github.com/nawka12/diffucore)** inference engine.

**Run the cells in order, top to bottom:**

1. **Config & Google Drive** — mounts Drive, creates the persistent folders
2. **Install** — clones Diffucore UI + engine, installs deps (~3–5 min, every session)
3. **Download Anima** — pulls the DiT / text encoder / VAE into Drive (first run only, ~5 GB, **16-connection parallel downloads via aria2**)
4. **Civitai / HF importer** — paste download URLs to add LoRAs or new checkpoints, **anytime, even mid-session**
5. **Launch** — starts the UI and prints your public **ngrok** URL (works from your phone)
6. **⚡ Preload + speed flags** — loads Anima with the Turing speed stack (fp16 accumulate, fp16 VAE, int8 attention + fused glue, resident encoders), arms **optimal generation defaults** (dpmpp_2m/beta · 20 steps · CFG interval) and runs a throwaway warmup image so your first click starts instantly — ~1.4–2× faster out of the box
7. **Status / stop** — utilities

**Persistence model (Google Drive):**
- `MyDrive/AnimaDiffucore/models/…` — checkpoints, DiTs, VAEs, text encoders, **LoRAs**, detailers (survive sessions)
- `MyDrive/AnimaDiffucore/outputs/YYYY-MM-DD/…` — every generated PNG with full metadata (survives sessions)
- Engine/UI code and pip installs are ephemeral and re-run each session (that's the Colab way)

**Requirements:** a free [ngrok](https://dashboard.ngrok.com/get-started/your-authtoken) authtoken (put it in the launch cell or in Colab Secrets as `NGROK_TOKEN`), and optionally a [civitai.com](https://civitai.com) API token for gated models.

> 💡 On a free **T4** expect roughly **20–40 s per image** at 832×1216 with the speed stack + 20-step defaults (~8–15 s with the turbo LoRA at 8–12 steps). The warmup image in cell 6 absorbs the first-image kernel warmup for you.
>
> ⚠️ The ngrok URL is **public** — anyone who has the link can use your session. Set a password in the launch cell to gate it.

### 1️⃣ Config & Google Drive

In [ ]:
# @title 1️⃣ Config & Google Drive {"form-width":"30%"}

USE_DRIVE = True  # @param {type:"boolean"}
DRIVE_FOLDER = "AnimaDiffucore"  # @param {type:"string"}

import os
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        print("Mounting Google Drive ...")
        drive.mount("/content/drive")
    GDRIVE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
else:
    GDRIVE = Path("/content/anima_local")  # ephemeral, wiped when the VM resets

# These two persist across sessions: models/ holds checkpoints + LoRAs,
# outputs/ is where the UI saves every generated PNG automatically.
DRIVE_MODELS = GDRIVE / "models"
DRIVE_OUTPUTS = GDRIVE / "outputs"
for sub in ("checkpoints", "diffusion-models", "vae", "text-encoders",
            "loras", "detailers", "upscalers", "oss_cache", "teacache_cache"):
    (DRIVE_MODELS / sub).mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)

UI_DIR = Path("/content/diffucore-ui")   # engine + UI code (re-cloned each session)
PORT = 7860

print("Drive root   :", GDRIVE)
print("Models folder:", DRIVE_MODELS)
print("Outputs      :", DRIVE_OUTPUTS)

### 2️⃣ Install Diffucore UI + engine

In [ ]:
# @title 2️⃣ Install Diffucore UI + engine (~3–5 min, run every session) {"form-width":"30%"}

FORCE_REINSTALL = False  # @param {type:"boolean"}

import subprocess, sys
from pathlib import Path

if FORCE_REINSTALL or not (UI_DIR / "backend" / "app.py").exists():
    !rm -rf $UI_DIR
    !git clone --recurse-submodules https://github.com/nawka12/diffucore-ui.git $UI_DIR
else:
    !git -C $UI_DIR pull --ff-only || true
!git -C $UI_DIR submodule update --init --recursive || true

# Torch: Colab already ships a CUDA build — repair it only if broken.
probe = subprocess.run(
    [sys.executable, "-c",
     "import torch; print('CUDA_OK' if torch.cuda.is_available() else 'CUDA_BAD')"],
    capture_output=True, text=True)
if "CUDA_OK" in probe.stdout:
    pass
elif "ModuleNotFoundError" in (probe.stdout + probe.stderr):
    print("Installing the CUDA build of torch (cu124) ...")
    !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu124
else:
    print("⚠️ torch found but CUDA is unavailable — switch the runtime to GPU")
    print("   (Runtime → Change runtime type → T4 GPU) for usable speeds.")

print("Installing Diffucore UI dependencies ...")
!pip install -q -r $UI_DIR/requirements.txt
!pip install -q -e $UI_DIR/diffucore
!pip install -q pyngrok ninja  # pyngrok = tunnel; ninja = builds the T4 int8 attention kernel

# aria2 = 16-connection parallel downloads (big first-run win on Colab).
# The downloader falls back to single-stream requests when aria2 is missing.
!apt-get -qq update -y >/dev/null 2>&1; apt-get -qq install -y aria2 >/dev/null 2>&1 && echo "aria2 ready (parallel downloads)" || echo "aria2 unavailable - single-stream downloads" 

# Point the app at Google Drive: models/ and outputs/ become symlinks, so every
# downloaded model, LoRA and generated image lands in Drive automatically.
import shutil
for link, target in ((UI_DIR / "models", DRIVE_MODELS), (UI_DIR / "outputs", DRIVE_OUTPUTS)):
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        shutil.rmtree(link, ignore_errors=True)
    link.symlink_to(target)

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("Install complete — models/ and outputs/ now live in", GDRIVE)

### 3️⃣ Download the Anima model files (into Drive)

In [ ]:
# @title 3️⃣ Download the Anima model (first run / new DiT — saved to Drive) {"form-width":"30%"}

DOWNLOAD_TURBO_DIT = False    # @param {type:"boolean"}
DOWNLOAD_TURBO_LORA = True    # @param {type:"boolean"}
INSTALL_FACE_DETAILER = True  # @param {type:"boolean"}

import re, time, shutil, subprocess, requests
from pathlib import Path
from tqdm.auto import tqdm

HF_ANIMA = "https://huggingface.co/circlestone-labs/Anima/resolve/main"
HF_ANIMA_LORAS = "https://huggingface.co/circlestone-labs/Anima-Official-LoRAs/resolve/main"
HF_ADETAILER = "https://huggingface.co/Bingsu/adetailer/resolve/main"

def _remote_info(url: str):
    """(content-disposition, content-length) via HEAD; ('', 0) on failure."""
    try:
        r = requests.head(url, allow_redirects=True, timeout=30)
        return (r.headers.get("Content-Disposition", ""),
                int(r.headers.get("Content-Length") or 0))
    except Exception:
        return "", 0


def _resolve_filename(url: str, content_disposition: str) -> str:
    m = re.search(r'filename="?([^";]+)"?', content_disposition)
    if m:
        return m.group(1)
    # Some CDNs answer HEAD without Content-Disposition: peek at a GET's headers.
    try:
        r = requests.get(url, stream=True, allow_redirects=True, timeout=30)
        cd = r.headers.get("Content-Disposition", "")
        r.close()
        m = re.search(r'filename="?([^";]+)"?', cd)
        if m:
            return m.group(1)
    except Exception:
        pass
    return url.split("?")[0].rstrip("/").split("/")[-1] or "download.bin"


def _aria2c(url: str, part: Path) -> bool:
    """Parallel download with aria2 (-c resumes the .part file). True on success."""
    if shutil.which("aria2c") is None:
        return False
    cmd = ["aria2c", "-c", "-x16", "-s16", "-k", "1M", "--file-allocation=none",
           "--console-log-level=warn", "--summary-interval=10",
           "--retry-wait=3", "--max-tries=10",
           "--timeout=60", "--connect-timeout=30",
           "-d", str(part.parent), "-o", part.name, url]
    try:
        rc = subprocess.run(cmd).returncode
    except Exception:
        return False
    return rc == 0 and part.exists() and part.stat().st_size > 0


def _stream_to(r, part, resume, total, filename):
    with open(part, "ab" if resume else "wb") as f, tqdm(
            total=total or None, initial=resume, unit="B",
            unit_scale=True, desc=filename) as bar:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk)
            bar.update(len(chunk))


def fetch(url: str, dest_dir, filename=None) -> Path:
    """Downloader: aria2 16-connection parallel when available, else streaming
    requests. Both resume partial .part files and skip files already on disk."""
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    cd, expected = _remote_info(url)
    if filename is None:
        filename = _resolve_filename(url, cd)
    final = dest_dir / filename
    part = dest_dir / (filename + ".part")
    if final.exists() and (expected == 0 or final.stat().st_size == expected):
        print(f"\u2713 already present: {filename}")
        return final
    if _aria2c(url, part):
        part.rename(final)
        print(f"\u2713 downloaded: {filename}")
        return final
    last_err = None
    for attempt in range(3):
        try:
            resume = part.stat().st_size if part.exists() else 0
            if resume and expected and resume == expected:
                part.rename(final)
                print(f"\u2713 downloaded: {filename}")
                return final
            if resume and expected and resume > expected:
                part.unlink(missing_ok=True)
                resume = 0
            r = requests.get(url, headers={"Range": f"bytes={resume}-"} if resume else {},
                             stream=True, timeout=(15, 60), allow_redirects=True)
            try:
                if resume and r.status_code == 416:
                    # Range start past EOF: drop the stale .part, restart clean.
                    part.unlink(missing_ok=True)
                    r.close()
                    continue
                if resume and r.status_code == 206:
                    total = resume + int(r.headers.get("Content-Length") or 0)
                else:
                    r.raise_for_status()
                    if resume:
                        print(f"{filename}: server ignored Range, restarting")
                        part.unlink(missing_ok=True)
                        resume = 0
                    total = expected or int(r.headers.get("Content-Length") or 0)
                _stream_to(r, part, resume, total, filename)
            finally:
                r.close()
            part.rename(final)
            print(f"\u2713 downloaded: {filename}")
            return final
        except Exception as e:
            last_err = e
            print(f"retry {attempt + 1}/3 failed: {e}")
            time.sleep(5)
    raise RuntimeError(f"download failed: {url} ({last_err})")

JOBS = [
    (f"{HF_ANIMA}/split_files/diffusion_models/anima-base-v1.0.safetensors",
     DRIVE_MODELS / "diffusion-models", "anima-base-v1.0.safetensors"),
    (f"{HF_ANIMA}/split_files/text_encoders/qwen_3_06b_base.safetensors",
     DRIVE_MODELS / "text-encoders", "qwen_3_06b_base.safetensors"),
    (f"{HF_ANIMA}/split_files/vae/qwen_image_vae.safetensors",
     DRIVE_MODELS / "vae", "qwen_image_vae.safetensors"),
]
if DOWNLOAD_TURBO_DIT:
    # Official turbo DiT — few steps, CFG 1. (Skip the turbo LoRA if you use this.)
    JOBS.append((f"{HF_ANIMA}/split_files/diffusion_models/anima-turbo-v1.0.safetensors",
                 DRIVE_MODELS / "diffusion-models", "anima-turbo-v1.0.safetensors"))
if DOWNLOAD_TURBO_LORA:
    # Official turbo LoRA for the base DiT — enables fast 8–12 step, CFG 1 generation.
    JOBS.append((f"{HF_ANIMA_LORAS}/anima-turbo-lora-v0.2.safetensors",
                 DRIVE_MODELS / "loras", "anima-turbo-lora-v0.2.safetensors"))
if INSTALL_FACE_DETAILER:
    # YOLO face detector for Diffucore UI's built-in ADetailer-style face fix.
    JOBS.append((f"{HF_ADETAILER}/face_yolov8n.pt",
                 DRIVE_MODELS / "detailers", "face_yolov8n.pt"))

for url, dest, name in JOBS:
    fetch(url, dest, name)

print("\nAll Anima files are saved in Google Drive.")
print("In the UI: select the Anima family, pick the DiT / VAE / text-encoder dropdowns, hit Load.")

### 4️⃣ Import LoRAs / checkpoints from Civitai or HuggingFace

Run this cell **anytime — even mid-session while the UI is serving**: new files
appear in the UI dropdowns after a refresh, no restart needed (the engine scans
the model folders live).

In [ ]:
# @title 4️⃣ Import LoRAs / checkpoints from Civitai or HuggingFace {"form-width":"30%"}
# Paste one URL per line. Civitai links should look like either:
#   https://civitai.com/api/download/models/<versionId>?token=<optional>
#   https://civitai.com/models/<id>?modelVersionId=<versionId>
# Some models require a token: civitai.com → Account settings → API keys.

CIVITAI_TOKEN = ""  # @param {type:"string"}
FILE_URLS = ""      # @param {type:"string"}
DESTINATION = "loras"  # @param ["loras", "diffusion-models", "checkpoints", "vae", "text-encoders", "upscalers", "detailers"]
CUSTOM_FILENAME = ""  # @param {type:"string"}  (optional; only used for a single URL)

import re

assert "fetch" in globals(), "Run cell 3 once first — it defines the downloader."

lines = [l.strip() for l in FILE_URLS.splitlines() if l.strip() and not l.strip().startswith("#")]
assert lines, "Paste at least one download URL above."
if len(lines) > 1 and CUSTOM_FILENAME.strip():
    print("(CUSTOM_FILENAME applies to a single URL \u2014 ignoring it for multiple URLs.)")
    CUSTOM_FILENAME = ""
for url in lines:
    name = CUSTOM_FILENAME.strip() or None
    if "civitai.com" in url:
        if "/api/download/" not in url:
            m = re.search(r"modelVersionId=(\d+)", url)
            if not m:
                raise SystemExit(
                    f"Unsupported Civitai URL: {url}\nUse the download button's link "
                    "(civitai.com/api/download/models/... or ...?modelVersionId=...).")
            url = f"https://civitai.com/api/download/models/{m.group(1)}"
        token = CIVITAI_TOKEN.strip()
        if token and "token=" not in url:
            url += ("&" if "?" in url else "?") + "token=" + token
    fetch(url, DRIVE_MODELS / DESTINATION, name)

print(f"\nDone → {DRIVE_MODELS / DESTINATION}")
print("Refresh the Diffucore UI (F5) — new files appear in the dropdowns, no restart needed.")

### 5️⃣ Launch the UI + ngrok tunnel

In [ ]:
# @title 5️⃣ Launch the UI and open the ngrok tunnel {"form-width":"30%"}

NGROK_TOKEN = ""   # @param {type:"string"}  (or add a Colab Secret named NGROK_TOKEN)
NGROK_DOMAIN = ""  # @param {type:"string"}  (optional: your static ngrok domain)
UI_PASSWORD = ""   # @param {type:"string"}  (recommended: protects the public URL)

import sys, time, socket, subprocess
from pathlib import Path

assert "UI_DIR" in globals(), "Run cells 1 and 2 first."

def port_open(p, host="127.0.0.1"):
    with socket.socket() as s:
        return s.connect_ex((host, p)) == 0

# Restart clean if a previous instance is running.
subprocess.run("pkill -f 'backend/app.py'", shell=True)
time.sleep(2)

LOG = UI_DIR / "server.log"
log = open(LOG, "a")
cmd = [sys.executable, str(UI_DIR / "backend" / "app.py"), "--listen", "--port", str(PORT)]
if UI_PASSWORD.strip():
    cmd += ["--auth-token", UI_PASSWORD.strip()]
proc = subprocess.Popen(cmd, cwd=str(UI_DIR), stdout=log, stderr=subprocess.STDOUT)
print("Starting Diffucore UI (PID", proc.pid, ") ...")

t0 = time.time()
while time.time() - t0 < 240:
    if proc.poll() is not None:
        print("✗ Server exited early. Last log lines:")
        print("\n".join(LOG.read_text(errors="ignore").splitlines()[-40:]))
        raise SystemExit(1)
    if port_open(PORT):
        print("✓ UI ready on port", PORT)
        break
    time.sleep(2)
else:
    print("✗ Timed out waiting for the server. Last log lines:")
    print("\n".join(LOG.read_text(errors="ignore").splitlines()[-40:]))
    raise SystemExit(1)

# ---- ngrok tunnel ----
token = NGROK_TOKEN.strip()
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("NGROK_TOKEN")
    except Exception:
        token = ""
assert token, "Set NGROK_TOKEN above or add a Colab Secret named NGROK_TOKEN (get it at ngrok.com → Your Authtoken)."

from pyngrok import ngrok
ngrok.set_auth_token(token)
ngrok.kill()
kwargs = {}
if NGROK_DOMAIN.strip():
    kwargs["domain"] = NGROK_DOMAIN.strip()
url = ngrok.connect(PORT, "http", **kwargs).public_url

print()
print("=" * 62)
print("🌍  ANIMA / Diffucore UI:", url)
if UI_PASSWORD.strip():
    print("🔑  Login password:", UI_PASSWORD)
print("🖼️  Images auto-save to Drive:", DRIVE_OUTPUTS)
print("ℹ️  Cell 6 warms the GPU up-front; without it the first image is a slow one.")
print("=" * 62)

### 6️⃣ ⚡ Preload Anima with T4 speed flags

Loads the model for you **with the Turing speed stack enabled** (fp16
accumulation, fp16 VAE decode, int8 attention + fused glue when the
capability checks pass), so the UI opens with everything already loaded
and running at full speed.

Also arms the **fast generation defaults** (dpmpp_2m/beta · 20 steps · CFG
interval) and runs a throwaway **warmup image**, so your very first click
generates at full speed with nothing left to warm up.

In [ ]:
# @title 6️⃣ ⚡ Preload Anima with T4 speed flags {"form-width":"30%"}
PRELOAD = True                 # @param {type:"boolean"}
FAST_PARAMS = True             # @param {type:"boolean"}  optimal defaults: dpmpp_2m + beta @ 20 steps (~1/3 less sampling) + CFG interval (~10% less)
RESIDENT_ENCODERS = True       # @param {type:"boolean"}  keep TE+VAE on GPU (~0.3 s/generation saved); set False if you OOM above ~1536²
WARMUP = True                  # @param {type:"boolean"}  runs one throwaway 2-step image so your first click skips kernel-JIT/cuDNN warmup
FAST_TURING_KERNELS = True     # @param {type:"boolean"}  int8 attention + fused glue (~1.4–1.8× stacked); auto-skips if unsupported
USE_TORCH_COMPILE = False      # @param {type:"boolean"}  ~1.3–1.4× but 30–180 s warmup + incompatible with int8 attention
PRELOAD_DIT = "anima-base-v1.0.safetensors"   # @param {type:"string"}
PRELOAD_VAE = "qwen_image_vae.safetensors"    # @param {type:"string"}
PRELOAD_TE  = "qwen_3_06b_base.safetensors"   # @param {type:"string"}
PRELOAD_TIMEOUT = 600          # @param {type:"integer"}  seconds per attempt (first load builds kernels + reads ~5.5 GB from Drive)

import json, time, threading, queue
import requests

assert "proc" in globals() and proc.poll() is None, "Run cell 5 first — the server must be up."
BASE = f"http://127.0.0.1:{PORT}"
HDRS = {"Authorization": f"Bearer {UI_PASSWORD.strip()}"} if UI_PASSWORD.strip() else {}

def _api(path, **kw):
    return requests.get(BASE + path, headers=HDRS, timeout=15, **kw)

# The startup hash scan can delay the first API response.
deadline = time.time() + 180
while True:
    try:
        _api("/api/status"); break
    except Exception:
        if time.time() > deadline:
            raise SystemExit("Server API did not come up — check cell 5 output / server.log.")
        time.sleep(2)

models = _api("/api/models").json()

# Reuse the last successful load form (files chosen in a previous run) when present.
lf = models.get("load_form") or {}
PRELOAD_DIT = lf.get("dit") or PRELOAD_DIT
PRELOAD_VAE = lf.get("vae") or PRELOAD_VAE
PRELOAD_TE  = lf.get("te")  or PRELOAD_TE

missing = [n for n, key in ((PRELOAD_DIT, "dits"), (PRELOAD_VAE, "vaes"), (PRELOAD_TE, "tes"))
           if n not in models.get(key, [])]
if missing:
    raise SystemExit(f"Not on disk yet: {missing} — run cell 3 first, then re-run this cell.")

# ---- optimal generation defaults (seeded into the UI's Generate form) ----
if FAST_PARAMS:
    st = _api("/api/settings").json()
    st["cfg_interval_start"] = 0.1   # skip the negative pass outside 0.1–0.9
    st["cfg_interval_end"] = 0.9     # -> ~10% fewer forward passes (guide: opt-in speedup)
    st["gen_defaults"] = {
        "sampler": "dpmpp_2m",       # guide: stays clean at 16–20 steps where
        "scheduler": "beta",         # er_sde wants 24–30 for the same quality
        "steps": 20, "cfg": 4.0, "width": 832, "height": 1216, "shift": 3.0,
    }
    r = requests.post(BASE + "/api/settings", json=st, headers=HDRS, timeout=30)
    if r.status_code == 200:
        print("⚙ fast params armed: dpmpp_2m/beta · 20 steps · CFG 4.0 · 832×1216 · CFG interval 0.1–0.9")
        print("   UI tab already open? Refresh it (F5) once so the Generate form picks these up.")
    else:
        print(f"⚠ could not save fast params ({r.status_code}) — continuing with UI defaults")

# ---- capability-gated attempt ladder ----
int8_ok = bool(models.get("int8_available"))
fuse_ok = bool(models.get("fused_glue_available"))
if USE_TORCH_COMPILE:
    plan = [(False, False, True, "torch.compile + fp16 accumulate + fp16 VAE")]
elif FAST_TURING_KERNELS and int8_ok:
    plan = [("int8_turing", fuse_ok, False,
             "int8 attention + fp16 accumulate + fp16 VAE" + (" + fused glue" if fuse_ok else "")),
            ("sdpa", fuse_ok, False,
             "fp16 accumulate + fp16 VAE" + (" + fused glue" if fuse_ok else " (int8 kernel unavailable)")),
            ("sdpa", False, False, "stock SDPA (all Turing kernels off)")]
elif FAST_TURING_KERNELS and fuse_ok:
    plan = [("sdpa", True, False, "fused glue + fp16 accumulate + fp16 VAE"),
            ("sdpa", False, False, "stock SDPA")]
else:
    plan = [("sdpa", False, False, "fp16 accumulate + fp16 VAE (Turing kernels unavailable here)")]

# ---- SSE listener: captures the load job's done/error event ----
events = queue.Queue()
def _listen():
    try:
        with requests.get(BASE + "/api/events", headers=HDRS, stream=True,
                          timeout=(10, 3600)) as r:
            for line in r.iter_lines(decode_unicode=True):
                if line and line.startswith("data:"):
                    try:
                        events.put(json.loads(line[5:].strip()))
                    except ValueError:
                        pass
    except Exception:
        pass
threading.Thread(target=_listen, daemon=True).start()
time.sleep(1.5)  # let the stream connect before the first POST

def _wait_job(job_id, timeout, expect_load=True):
    """Wait for this job's SSE event (and, for loads, the status flip).
    Returns (ok, event_or_detail)."""
    t0 = time.time()
    while time.time() - t0 < timeout:
        if expect_load:
            try:
                st = _api("/api/status").json().get("status", "")
            except Exception:
                st = ""
            if "Anima(" in st:
                return True, st
        try:
            ev = events.get(timeout=2)
        except queue.Empty:
            continue
        if ev.get("type") == "error" and ev.get("job") == job_id:
            return False, f"load failed: {ev.get('message', '?')}"
        if ev.get("type") == "done" and ev.get("job") == job_id:
            if not expect_load:
                return True, ev
            time.sleep(1)
            try:
                st = _api("/api/status").json().get("status", "")
            except Exception:
                st = ""
            if "Anima(" in st:
                return True, st
    return False, ("timed out" if expect_load else "timed out")

if not PRELOAD:
    print("PRELOAD off — load the model from the UI top bar instead.")
else:
    payload_base = {"model_type": "Anima", "dit": PRELOAD_DIT, "vae": PRELOAD_VAE,
                    "te": PRELOAD_TE,
                    "offload": "none" if RESIDENT_ENCODERS else None,
                    "cuda_graphs": False, "channels_last": False, "tf32": False,
                    "fp16_accumulation": True, "vae_fp16": True}
    ok = False
    for attention, fused_glue, compile_, label in plan:
        payload = {**payload_base, "attention": attention,
                   "fused_glue": fused_glue, "compile": compile_}
        r = requests.post(BASE + "/api/load", json=payload, headers=HDRS, timeout=30)
        if r.status_code != 200:
            print(f"load rejected ({r.status_code}): {r.text[:200]}")
            continue
        job_id = r.json().get("job")
        print(f"⏳ Loading {PRELOAD_DIT} — {label} (up to {PRELOAD_TIMEOUT}s) ...")
        ok, detail = _wait_job(job_id, PRELOAD_TIMEOUT)
        if ok:
            print("⚡", detail)
            print(f"   speed stack: {label}" + (" · encoders resident" if RESIDENT_ENCODERS else ""))
            break
        print(f"   ↳ {detail}; falling back to safer flags")

    # ---- warmup: absorb kernel JIT / cuDNN autotune before the user's first click ----
    if ok and WARMUP:
        warm = {"mode": "t2i", "prompt": "a plain gray cube on a white table, soft studio light",
                "neg": "", "sampler": "dpmpp_2m", "scheduler": "beta", "steps": 2, "cfg": 4.0,
                "seed": 0, "width": 832, "height": 1216, "shift": 3.0,
                "preview": False, "blur_check": False}
        try:
            r = requests.post(BASE + "/api/generate", json=warm, headers=HDRS, timeout=30)
            if r.status_code == 200:
                t0 = time.time()
                wok, ev = _wait_job(r.json().get("job"), PRELOAD_TIMEOUT, expect_load=False)
                if wok and isinstance(ev, dict) and ev.get("path"):
                    print(f"🔥 warmup done in {time.time() - t0:.1f}s (throwaway image removed)")
                    try:
                        requests.delete(BASE + "/api/gallery",
                                        params={"path": ev["path"]}, headers=HDRS, timeout=15)
                    except Exception:
                        pass
                else:
                    print(f"⚠ warmup skipped ({ev}) — generation will warm up on your first image")
            else:
                print(f"⚠ warmup skipped ({r.status_code}) — generation will warm up on your first image")
        except Exception as e:
            print(f"⚠ warmup skipped ({e}) — generation will warm up on your first image")

    if not ok:
        print("✗ Preload did not finish. Last server log lines:")
        print("\n".join((UI_DIR / "server.log").read_text(errors="ignore").splitlines()[-40:]))
        print("Load manually from the UI top bar — speed flags sit next to the Load button.")

### 7️⃣ Server utilities

In [ ]:
# @title 7️⃣ Server status / stop {"form-width":"30%"}

import subprocess

ACTION = "status"  # @param ["status", "stop server"]

if ACTION == "status":
    running = "proc" in globals() and proc.poll() is None
    print("Server:", f"running (PID {proc.pid})" if running else "stopped")
    if running:
        print("\n--- last server log lines ---")
        print("\n".join((UI_DIR / "server.log").read_text(errors="ignore").splitlines()[-15:]))
    try:
        from pyngrok import ngrok
        for t in ngrok.get_tunnels():
            print("Tunnel:", t.public_url)
    except Exception:
        pass
elif ACTION == "stop server":
    subprocess.run("pkill -f 'backend/app.py'", shell=True)
    print("Server stopped. (Run cell 5 again to relaunch.)")

---
## 📋 Cheat sheet

**Loading a model (top bar):** pick **Anima** → choose
`anima-base-v1.0.safetensors` (DiT) + `qwen_image_vae.safetensors` (VAE) +
`qwen_3_06b_base.safetensors` (text encoder) → **Load**.

**Settings that work well with Anima (cell 6 pre-fills the fast base-model values):**

| | Base model | Turbo (LoRA or DiT) |
|---|---|---|
| Sampler / scheduler | `dpmpp_2m` / `beta` @ 20 steps (pre-filled by cell 6) — or `er_sde` / `flow` for max quality | `euler` / `flow` |
| Steps | 20 (fast) / 28–32 (`er_sde` quality) | 8–12 |
| CFG | 4.0 | 1.0 |
| Resolution | 832×1216, 1024×1024 | same |

- **Write long, natural-language prompts** — Anima is LLM-conditioned (Qwen3 + T5), tag-style prompts give weak results.
- **LoRAs:** just type them into the prompt, e.g. `a castle in autumn, <lora:my-lora:0.8>`.
- **img2img:** keep strength low (~0.2–0.4) — Anima's flow schedule erases structure fast.
- **Inpaint:** use denoise ~0.9–1.0 for a clean repaint.
- **Face fix:** toggle the **Detailer** (uses the `face_yolov8n.pt` you downloaded).

**Files on your Drive** (`MyDrive/AnimaDiffucore/`):
- `models/diffusion-models/` `models/checkpoints/` `models/loras/` `models/vae/` `models/text-encoders/` `models/detailers/` `models/upscalers/`
- `outputs/YYYY-MM-DD/NNNNN-seed.png` — every generation with full PNG metadata (browsable in the UI gallery too)

**⚡ Speed (pre-configured by cell 6)**
- **Optimal parameters seeded into the form:** `dpmpp_2m` + `beta` @ **20 steps**, CFG 4.0, 832×1216 — the guide's low-step pairing (clean at 16–20 steps where `er_sde` needs 24–30) = **~1/3 less sampling time** per image
- **CFG guidance interval 0.1–0.9** — the negative-prompt pass is skipped outside the band → ~10% fewer forward passes (adjustable in Settings → Sampler defaults)
- **Resident encoders** — TE + VAE stay on the GPU (~0.3 s saved per image). If a generation ever OOMs above ~1536², re-run with `RESIDENT_ENCODERS = False`
- **Warmup image** — cell 6 generates (then removes) one 2-step image so kernel JIT + cuDNN autotune happen before *your* first click
- **fp16 accumulation** (~1.17×) + **fp16 VAE decode** (~2.8× decode) — always on via the preload cell
- **int8 attention + fused glue** on T4's Turing chip — stacks to roughly **1.4–1.8× end-to-end** vs stock (measured on the same sm75 architecture class); auto-skips if the kernel toolchain is missing
- Downloads use **aria2, 16 connections per file** (falls back to single-stream automatically)
- `USE_TORCH_COMPILE = True` is an alternative ~1.3–1.4× (30–180 s one-time warmup, incompatible with int8 attention)
- The int8/fused-glue kernels rebuild once per session — that happens *inside* the preload cell, before you open the UI
- For more: **TeaCache** stays an opt-in quality/speed trade-off in Settings (off by default); the **CFG guidance interval** is already armed by cell 6 — reset it to 0.0–1.0 in Settings to turn it off

**Tips**
- Mid-session model import: cell 4 → paste URL → run → F5 in the UI. The engine picks up new files live.
- ngrok free shows an interstitial "Visit Site" page once per link — click through.
- If the runtime idles out, re-run cells 1, 2, 5 and 6 (models and images are safe in Drive).
- For fastest back-to-back runs, turn **off** the page's *Blur NSFW* toggle — otherwise a ~0.5 s background rating job can sit between generations (images then aren't auto-rated).
- Colab may warn about the notebook not being authored by Google — that's normal for any third-party notebook.